# Adversarial examples & attacks

An adversarial attack finds a tiny input change that moves the score across the model boundary.

FGSM uses the input-gradient sign of cross-entropy. Here the same real tabular attack is applied to every rung, including real Wine and Breast Cancer data. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: FGSM

The attack is $$x_{adv}=x+\varepsilon\,\operatorname{sign}(\nabla_x \ell(x,y)).$$ The code computes the true gradient of multinomial logistic cross-entropy with respect to standardized tabular inputs.

In [ ]:

components = np.array([0.4, 0.3, 0.2], dtype=float)
knob = 0.1
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 0.9)
assert np.isclose(absolute_total, 0.9)
assert np.isclose(round(guarded, 3), 0.99)
assert np.isclose(round(leading_share, 3), 0.444)
assert np.isclose(knob, 0.100)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


For a linear softmax model, $\nabla_x\ell=(p-y_{onehot})W$. FGSM keeps only the sign, so the declared $\varepsilon$ is the essential unit of the robustness claim.

In [ ]:

X, y = clf_ladder()[1][1:]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
clean_accuracy = accuracy_score(y_test, model.predict(x_test))
attacked = fgsm_attack(model, x_test, y_test, eps=0.10)
adversarial_accuracy = accuracy_score(y_test, model.predict(attacked))
print("clean_accuracy", round(clean_accuracy, 3))
print("adversarial_accuracy_eps_0_10", round(adversarial_accuracy, 3))
print("mean_linf_change", round(float(np.max(np.abs(attacked - x_test), axis=1).mean()), 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same attack across D1-D5

The metric is robust accuracy under a declared FGSM radius.

In [ ]:

epsilon = 0.15
attack_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    clean_acc = accuracy_score(y_test, model.predict(x_test))
    robust_acc = robust_accuracy_for_eps(model, x_test, y_test, epsilon)
    attack_results.append({"rung": rung, "name": name, "clean_acc": clean_acc, "robust_acc": robust_acc})
print("rung | clean_acc | robust_acc_eps_0_15")
for row in attack_results:
    print(row["rung"], round(row["clean_acc"], 3), round(row["robust_acc"], 3))


## Results visualization

Left: clean versus attacked projections per rung. Right: robust accuracy versus epsilon.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row, data in zip(axes, attack_results, clf_ladder()):
    name, X, y = data
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    attacked = fgsm_attack(model, x_test, y_test, epsilon)
    plot_2d_projection(ax, attacked, model.predict(attacked), f"D{row['rung']} attacked")
plt.tight_layout()
plt.show()

eps_grid = np.linspace(0.0, 0.30, 7)
fig, ax = plt.subplots(figsize=(7, 4))
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    curve = [robust_accuracy_for_eps(model, x_test, y_test, eps) for eps in eps_grid]
    ax.plot(eps_grid, curve, marker="o", label=f"D{rung}")
ax.set_xlabel("FGSM epsilon in standardized feature units")
ax.set_ylabel("robust accuracy")
ax.legend()
plt.show()


## Pitfall on D5: reporting accuracy without $\varepsilon$

The wrong number is a single clean accuracy. The fix reports the whole declared norm/radius ladder.

In [ ]:

name, X, y = clf_ladder()[-1]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
wrong_clean_only = accuracy_score(y_test, model.predict(x_test))
print("wrong_clean_only", round(wrong_clean_only, 3))
for eps in np.linspace(0.0, 0.30, 7):
    score = robust_accuracy_for_eps(model, x_test, y_test, eps)
    print("linf_eps", round(float(eps), 2), "robust_accuracy", round(score, 3))


## Evaluate it + Practice

- Metric: robust accuracy under FGSM epsilon.
- No-skill baseline: clean held-out accuracy at epsilon zero.
- Cheap sanity check: epsilon zero must equal clean accuracy.
- Ablation: replace sign gradient with random signs and compare attack strength.
- Failure signals: accuracy is quoted without norm, radius, preprocessing, or label target.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Try an untargeted attack with half the epsilon and compare rank order across rungs.

3. Check whether clipping standardized features changes the D5 result.